# PG-LIF — P1 Consolidation: decisive Regime A comparison under the benchmark protocol
**Context.** The fix sweep confirmed the leak-amplification mechanism (scaled drive: 10% → 37%) but left two confounds: the κ = 0 control was never run past 5 epochs (it hit 35.4% there, fastest of all), and the sweep lacked the LR scheduler used in the main benchmark, causing ±10 pp oscillations and making numbers incomparable to TC-LIF's 75.1%.

This notebook removes both confounds: **four configurations, 20 epochs each, exact benchmark protocol** (Adam 5e-4, StepLR ×0.1 at epoch 12, clip 5, batch 64, T = 100), directly comparable to the stored baseline numbers (TC-LIF 75.1, DH-LIF 70.3, ALIF 47.6, LIF 18.7):

1. `kappa0 (ablation)` — plateau disconnected; the bar to beat.
2. `scaled` — κ·p·(1−αm), β = 1 (Stage-1 winner).
3. `beta0` — unscaled drive, adaptation off (Stage-2 winner).
4. `scaled+beta0.5` — scaled drive with halved adaptation (untested middle ground).

**Decision rule (pre-registered):** if the best plateau configuration does not clearly exceed `kappa0` (≥ 3 pp), the Regime A memory claim is not supported on SHD at this scale; the project pivots to the P3 learning-gate claim and the manuscript is reframed accordingly — as the plan's risk register already provides. Runtime ~40 min on a T4.

In [ ]:
import os, json, time, math
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/My Drive'
    if not os.path.isdir(ROOT): ROOT = '/content/drive/MyDrive'
    BASE = os.path.join(ROOT, 'PG_LIF')
except Exception:
    BASE = './PG_LIF'
DATA = os.path.join(BASE, 'data', 'SHD')
OUT = os.path.join(BASE, 'P1_results', 'consolidation_' + time.strftime('%Y%m%d_%H%M%S'))
os.makedirs(OUT, exist_ok=True)
T_BINS, MAX_TIME, N_IN, N_OUT, HIDDEN, BATCH, LR = 100, 1.4, 700, 20, 128, 64, 5e-4
print('Consolidation folder:', OUT)

In [ ]:
import numpy as np, h5py, torch, torch.nn as nn
def load_split(fname):
    with h5py.File(os.path.join(DATA, fname), 'r') as f:
        return ([np.array(t) for t in f['spikes']['times']],
                [np.array(u) for u in f['spikes']['units']],
                np.array(f['labels'], dtype=np.int64))
TR = load_split('shd_train.h5'); TE = load_split('shd_test.h5')
def batches(split, batch_size, shuffle, device='cpu'):
    times, units, labels = split
    idx = np.random.permutation(len(labels)) if shuffle else np.arange(len(labels))
    for b0 in range(0, len(idx), batch_size):
        sel = idx[b0:b0+batch_size]
        x = torch.zeros(len(sel), T_BINS, N_IN)
        for i, j in enumerate(sel):
            tt = times[j]; uu = units[j]; keep = tt < MAX_TIME
            tb = np.clip((tt[keep] / MAX_TIME * T_BINS).astype(int), 0, T_BINS-1)
            x[i, tb, uu[keep]] = 1.0
        yield x.to(device), torch.as_tensor(labels[sel]).to(device)
print('train', len(TR[2]), '| test', len(TE[2]))

In [ ]:
class Triangle(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x); return (x >= 0).float()
    @staticmethod
    def backward(ctx, g):
        (x,) = ctx.saved_tensors
        return g * torch.clamp(1.0 - x.abs(), min=0.0)
spike_fn = Triangle.apply
def decay(tau): return math.exp(-1.0 / tau)

class PGLIFCell(nn.Module):
    def __init__(self, N, beta=1.0, kappa0=1.0, tau_p=None, theta_d=1.0, tref_p=10,
                 scaled=False, highpass=False, kappa_fixed_zero=False):
        super().__init__(); self.N = N
        self.am = decay(20); self.ad = decay(20); self.aa = decay(200); self.ahp = decay(200)
        tau_p = tau_p or T_BINS / 2
        ap0 = decay(tau_p)
        self.ap_logit = nn.Parameter(torch.full((N,), math.log(ap0 / (1 - ap0))))
        self.kappa = nn.Parameter(torch.full((N,), float(kappa0)))
        self.beta, self.th, self.th_d, self.P0, self.tref_p = beta, 1.0, theta_d, 1.0, tref_p
        self.scaled, self.highpass, self.kzero = scaled, highpass, kappa_fixed_zero
    def init(self, B, dev):
        z = lambda: torch.zeros(B, self.N, device=dev)
        self.vs, self.vd, self.p, self.a, self.pbar = z(), z(), z(), z(), z()
        self.rp = torch.zeros(B, self.N, device=dev)
    def forward(self, I_ff, I_rec):
        self.vd = self.ad * self.vd + I_ff
        ed = spike_fn(self.vd - self.th_d) * (self.rp == 0).float()
        self.rp = torch.clamp(self.rp - 1, min=0) + ed.detach() * self.tref_p
        self.p = torch.sigmoid(self.ap_logit) * self.p + self.P0 * ed
        if self.kzero:
            drive = 0.0
        elif self.highpass:
            self.pbar = self.ahp * self.pbar + (1 - self.ahp) * self.p.detach()
            drive = self.kappa * (self.p - self.pbar)
        elif self.scaled:
            drive = self.kappa * self.p * (1 - self.am)
        else:
            drive = self.kappa * self.p
        self.vs = self.am * self.vs + I_ff + I_rec + drive
        th = self.th + self.beta * self.a
        s = spike_fn(self.vs - th)
        self.vs = self.vs - s.detach() * th.detach()
        self.a = self.aa * self.a + s.detach()
        return s

class RecSNN(nn.Module):
    def __init__(self, **kw):
        super().__init__()
        self.w_in = nn.Linear(N_IN, HIDDEN); self.w_rec = nn.Linear(HIDDEN, HIDDEN, bias=False)
        self.cell = PGLIFCell(HIDDEN, **kw); self.w_out = nn.Linear(HIDDEN, N_OUT)
        self.a_out = decay(20); nn.init.orthogonal_(self.w_rec.weight)
    def forward(self, x):
        B, T, _ = x.shape; self.cell.init(B, x.device)
        s = torch.zeros(B, HIDDEN, device=x.device)
        out = torch.zeros(B, N_OUT, device=x.device); vo = torch.zeros(B, N_OUT, device=x.device)
        for t in range(T):
            s = self.cell(self.w_in(x[:, t]), self.w_rec(s))
            vo = self.a_out * vo + self.w_out(s); out = out + vo
        return out

def accuracy(model, split, device, limit=1024):
    model.eval(); correct = tot = 0
    with torch.no_grad():
        for x, y in batches(split, 256, shuffle=False, device=device):
            out = model(x); correct += (out.argmax(1) == y).sum().item(); tot += len(y)
            if tot >= limit: break
    return correct / tot

def train_cfg(tag, epochs, resume=None, **kw):
    torch.manual_seed(0); np.random.seed(0)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    m = RecSNN(**kw).to(device)
    o = torch.optim.Adam(m.parameters(), lr=LR); cr = nn.CrossEntropyLoss()
    sch = torch.optim.lr_scheduler.MultiStepLR(o, milestones=[12], gamma=0.1)
    hist = []
    if resume is not None:
        m.load_state_dict(torch.load(resume['ckpt'])); hist = resume['history']
        o.load_state_dict(torch.load(resume['opt']))
        for _ in range(len(hist)): sch.step()
    for ep in range(len(hist), epochs):
        m.train(); t0 = time.time()
        for x, y in batches(TR, BATCH, shuffle=True, device=device):
            o.zero_grad(); cr(m(x), y).backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 5.0); o.step()
        sch.step()
        hist.append({'epoch': ep, 'train_acc': accuracy(m, TR, device),
                     'test_acc': accuracy(m, TE, device)})
        print(f"{tag:18s} ep{ep:02d} train {hist[-1]['train_acc']:.3f} test {hist[-1]['test_acc']:.3f} {time.time()-t0:.0f}s")
    ck = os.path.join(OUT, tag.replace(' ', '_') + '.pt'); ok = ck.replace('.pt', '_opt.pt')
    torch.save(m.state_dict(), ck); torch.save(o.state_dict(), ok)
    return {'tag': tag, 'kw': {k: str(v) for k, v in kw.items()}, 'history': hist,
            'ckpt': ck, 'opt': ok}

## Four configurations, 20 epochs, exact benchmark protocol

In [ ]:
CONFIGS = [
  ('kappa0 (ablation)', dict(kappa_fixed_zero=True)),
  ('scaled',            dict(scaled=True)),
  ('beta0',             dict(beta=0.0)),
  ('scaled+beta0.5',    dict(scaled=True, beta=0.5)),
]
RUNS = [train_cfg(tag, 20, **kw) for tag, kw in CONFIGS]
json.dump([{k: r[k] for k in ('tag', 'kw', 'history')} for r in RUNS],
          open(os.path.join(OUT, 'consolidation.json'), 'w'), indent=2)
print()
for r in RUNS:
    print(f"{r['tag']:20s} best test {max(h['test_acc'] for h in r['history']):.3f}  "
          f"final train {r['history'][-1]['train_acc']:.3f}")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 5))
for r in RUNS:
    plt.plot([h['epoch'] for h in r['history']], [h['test_acc'] for h in r['history']], lw=2, label=r['tag'])
for name, val in [('TC-LIF', 0.7513), ('DH-LIF', 0.7027), ('ALIF', 0.4761), ('LIF', 0.1873)]:
    plt.axhline(val, ls=':', lw=1)
    plt.text(19.4, val + 0.005, name, fontsize=7, ha='right')
plt.xlabel('epoch'); plt.ylabel('SHD test accuracy'); plt.legend(fontsize=9)
plt.title('P1 consolidation - benchmark protocol, 20 epochs')
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'fig_consolidation.png'), dpi=300); plt.show()

k0 = max(h['test_acc'] for h in RUNS[0]['history'])
best = max(RUNS[1:], key=lambda r: max(h['test_acc'] for h in r['history']))
bv = max(h['test_acc'] for h in best['history'])
print(f"kappa0 ablation best: {k0:.3f}")
print(f"best plateau config:  {best['tag']} at {bv:.3f}  (margin {100*(bv-k0):+.1f} pp)")
if bv - k0 >= 0.03:
    print("DECISION: plateau exceeds its ablation -> adopt this config as v3 and run the full P1 benchmark.")
else:
    print("DECISION: plateau does NOT clearly exceed its ablation on SHD Regime A at this scale.")
    print("Per the pre-registered rule and the plan's risk register: pivot to the P3 learning-gate claim;")
    print("Regime A enters the paper as an honest ablation finding, not a headline result.")

### After this run
Upload the executed notebook. Depending on the decision line: either P1 v3 goes to the full multi-seed benchmark with the selected config, or we reframe — the manuscript's contribution list shifts weight to the plateau-as-learning-gate claim (P3), the theory section keeps the gradient analysis as a property rather than a performance claim, and the Regime A result is reported as what it is. Either branch is publishable; only one is honest to whichever data we get.